# DistilBERT

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "distilbert-base-uncased"   # for RoBERTa later: "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model.eval()

text = "WASHINGTON (Reuters) - The U.S. Senate voted on Tuesday to approve the budget."
inputs = tokenizer(text, truncation=True, max_length=256, return_tensors="pt")

print(inputs["input_ids"].shape)
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])[:12])

with torch.no_grad():
    logits = model(**inputs).logits
probs = torch.softmax(logits, dim=1)
print("logits:", logits)
print("prob_real, prob_fake:", probs)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\VIVEK\OneDrive\Desktop\AIML\Internship_Fake_detection\Fake_News_Misinformation_Detection\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VIVEK\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 21])
['[CLS]', 'washington', '(', 'reuters', ')', '-', 'the', 'u', '.', 's', '.', 'senate']
logits: tensor([[0.0890, 0.0653]])
prob_real, prob_fake: tensor([[0.5059, 0.4941]])


# Fine Tuning

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODEL = "distilbert-base-uncased"

# same data and split as the baseline
df = pd.read_csv("../data/interim/master.csv", usecols=["row_id", "label", "raw_text"])
df = df.merge(pd.read_csv("../data/splits/split.csv"), on="row_id")
train = df[df["split"] == "train"].sample(500, random_state=42)
val = df[df["split"] == "val"].sample(200, random_state=42)

tok = AutoTokenizer.from_pretrained(MODEL)

def to_ds(d):
    ds = Dataset.from_dict({"text": d["raw_text"].tolist(), "label": d["label"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)

train_ds, val_ds = to_ds(train), to_ds(val)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

args = TrainingArguments(
    output_dir="../models/transformer/smoke_test",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    seed=42,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\VIVEK\OneDrive\Desktop\AIML\Internship_Fake_detection\Fake_News_Misinformation_Detection\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accel

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.304791,0.995000,0.995434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\VIVEK\OneDrive\Desktop\AIML\Internship_Fake_detection\Fake_News_Misinformation_Detection\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\VIVEK\OneDrive\Desktop\AIML\Internship_Fake_detection\Fake_News_Misinformation_Detection\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.304791,1,0.995000,0.995434


{'eval_loss': 0.30479124188423157, 'eval_accuracy': 0.995, 'eval_f1': 0.9954337899543378}


: 